# 01 Data Cleaning
E-Commerce Sales & Customer Analytics project.
This notebook inspects the raw data, documents every quality issue found,
and explains the reasoning behind each cleaning decision.

In [3]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

customers = pd.read_csv('../data/raw/customers.csv')
products = pd.read_csv('../data/raw/products.csv')
orders = pd.read_csv('../data/raw/orders.csv')
order_items = pd.read_csv('../data/raw/order_items.csv')

for name, df in [('customers', customers), ('products', products), ('orders', orders), ('order_items', order_items)]:
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

customers: 5,015 rows, 7 columns
products: 800 rows, 6 columns
orders: 25,000 rows, 6 columns
order_items: 54,078 rows, 5 columns


## Inspect Data

In [4]:
customers.info()
print()
print(customers.isnull().sum())
print()
print("Duplicate customer_id rows:", customers.duplicated(subset='customer_id').sum())
print(customers['age'].describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5015 entries, 0 to 5014
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  5015 non-null   int64 
 1   name         5015 non-null   object
 2   gender       4950 non-null   object
 3   age          5015 non-null   int64 
 4   city         5015 non-null   object
 5   state        5015 non-null   object
 6   signup_date  5015 non-null   object
dtypes: int64(2), object(5)
memory usage: 274.4+ KB

customer_id     0
name            0
gender         65
age             0
city            0
state           0
signup_date     0
dtype: int64

Duplicate customer_id rows: 15
count    5015.000000
mean       31.378465
std         9.164256
min        -1.000000
25%        25.000000
50%        32.000000
75%        37.000000
max        67.000000
Name: age, dtype: float64


In [5]:
products.info()
print()
print(products.isnull().sum())
print("Negative selling_price rows:", (products['selling_price'] < 0).sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   product_id     800 non-null    int64  
 1   product_name   800 non-null    object 
 2   category       800 non-null    object 
 3   brand          800 non-null    object 
 4   cost_price     800 non-null    float64
 5   selling_price  800 non-null    float64
dtypes: float64(2), int64(1), object(3)
memory usage: 37.6+ KB

product_id       0
product_name     0
category         0
brand            0
cost_price       0
selling_price    0
dtype: int64
Negative selling_price rows: 8


In [6]:
orders.info()
print()
print(orders.isnull().sum())
print("Duplicate order_id rows:", orders.duplicated(subset='order_id').sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        25000 non-null  int64  
 1   customer_id     25000 non-null  int64  
 2   order_date      25000 non-null  object 
 3   payment_method  24625 non-null  object 
 4   discount        25000 non-null  float64
 5   status          25000 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 1.1+ MB

order_id            0
customer_id         0
order_date          0
payment_method    375
discount            0
status              0
dtype: int64
Duplicate order_id rows: 0


In [7]:
order_items.info()
print()
print("Zero/negative quantity rows:", (order_items['quantity'] <= 0).sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54078 entries, 0 to 54077
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_item_id  54078 non-null  int64  
 1   order_id       54078 non-null  int64  
 2   product_id     54078 non-null  int64  
 3   quantity       54078 non-null  int64  
 4   unit_price     54078 non-null  float64
dtypes: float64(1), int64(4)
memory usage: 2.1 MB

Zero/negative quantity rows: 10


## Issues found and cleaning decisions

| Table | Issue | Decision | Reasoning |
|---|---|---|---|
| customers | Duplicate `customer_id` rows | Drop duplicates, keep first | Same customer entered twice (form double-submit); the record itself is not informative |
| customers | Missing `gender` | Fill with `'Unknown'` | Not recoverable from other fields; deleting would lose the customer's real orders |
| customers | Invalid `age` (negative) | Set to `NaN` | Not recoverable, but age isn't critical to core revenue analysis |
| products | Negative `selling_price` | Take absolute value | Sign error at entry, not a real negative price |
| orders | Missing `payment_method` | Fill with `'Unknown'` | Order and its revenue are still real; payment method is a secondary attribute |
| order_items | `quantity <= 0` | Drop the row | A zero-quantity line item isn't a real transaction — no informative value in keeping it |
| order_items | Orphaned `order_id`/`product_id` | Drop the row | Referential integrity — a line item pointing at a non-existent order/product can't be analyzed |

The full, reusable version of this logic lives in `src/data_cleaning.py` so it can be re-run any time the raw data changes.


In [8]:
import sys
sys.path.append('../src')
from data_cleaning import clean_customers, clean_products, clean_orders, clean_order_items

import os
os.chdir('..')  # so the relative paths inside data_cleaning.py resolve correctly

customers_clean = clean_customers()
products_clean = clean_products()
orders_clean = clean_orders()
order_items_clean = clean_order_items(
    valid_order_ids=set(orders_clean['order_id']),
    valid_product_ids=set(products_clean['product_id'])
)

[customers] removed 15 duplicate rows
[customers] final row count: 5,000
[products] removed 0 duplicate rows
[products] fixed 8 negative selling_price values
[products] final row count: 800
[orders] removed 0 duplicate rows
[orders] final row count: 25,000
[order_items] removed 10 rows (10 zero-quantity, 0 orphaned)
[order_items] final row count: 54,068


## Verify the cleaned data
Quick sanity check before moving on to SQL loading and EDA.


In [9]:
assert customers_clean['customer_id'].is_unique
assert orders_clean['order_id'].is_unique
assert (order_items_clean['quantity'] > 0).all()
assert order_items_clean['order_id'].isin(orders_clean['order_id']).all()
assert order_items_clean['product_id'].isin(products_clean['product_id']).all()
print("All cleaned tables pass referential-integrity and validity checks.")


All cleaned tables pass referential-integrity and validity checks.
